# 面试题：怎样从零实现 GIN，并用它做图分类？

## 可以直接复述的回答

GIN 用邻居求和而不是求均值，更新形式是 MLP((1+epsilon)×自身 + 邻居特征和)。sum 能保留邻居多重集合的计数信息，是 GIN 接近 Weisfeiler-Lehman 判别能力的关键。图分类通常堆叠若干消息层，再把全部节点表示池化为图向量并接分类头。训练与评估要按整张图切分，不能把同一图的节点随机分到两侧。置换不变性只说明节点编号变化不应改变输出，不等于模型能泛化到未见结构；两项必须分开评估。本题在 8 张五节点图上训练，再用 4 张六/七节点且拓扑未见的图诚实评估结构泛化，同时另做同一张图的节点重编号实验。最后用一邻居与两相同邻居复现 mean 聚合不可区分而 sum 可以区分的反例。

## 真实案例

每张图表示账户共享设备的局部网络；节点特征统一为账户存在标记，标签表示是否包含闭环团伙结构。训练集只有 5 节点图，结构泛化集只有 6/7 节点图，因此不可能只是训练拓扑的节点重标号。所有图都是脱敏教学构造，不能直接用于真实风控。

In [1]:
from pprint import pprint  # 导入结构化打印函数以展示图结构和分类结果。
import torch  # 导入 PyTorch 以实现真实 GIN 前向与反向传播。
torch.manual_seed(41)  # 固定参数初始化保证结果可复现。
torch.set_num_threads(1)  # 限制 CPU 线程以稳定小图训练。
graph_specs = [  # 定义八张五节点训练图和四张更大结构泛化图。
    {"id": "G1", "node_count": 5, "edges": [(0, 1), (1, 2), (2, 0), (2, 3), (3, 4)], "label": 1, "split": "train", "说明": "三账户闭环带尾链"},  # 定义五节点三角环训练正例。
    {"id": "G2", "node_count": 5, "edges": [(0, 1), (1, 2), (2, 3), (3, 0), (3, 4)], "label": 1, "split": "train", "说明": "四账户闭环"},  # 定义五节点四环训练正例。
    {"id": "G3", "node_count": 5, "edges": [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0)], "label": 1, "split": "train", "说明": "五账户闭环"},  # 定义五节点整环训练正例。
    {"id": "G4", "node_count": 5, "edges": [(0, 1), (1, 2), (2, 0), (0, 3), (0, 4)], "label": 1, "split": "train", "说明": "三角团伙带双叶"},  # 定义含两个叶节点的训练正例。
    {"id": "G5", "node_count": 5, "edges": [(0, 1), (1, 2), (2, 3), (3, 4)], "label": 0, "split": "train", "说明": "正常链路"},  # 定义五节点路径训练负例。
    {"id": "G6", "node_count": 5, "edges": [(0, 1), (0, 2), (0, 3), (0, 4)], "label": 0, "split": "train", "说明": "正常星形"},  # 定义五节点星形训练负例。
    {"id": "G7", "node_count": 5, "edges": [(0, 1), (1, 2), (1, 3), (3, 4)], "label": 0, "split": "train", "说明": "正常树形"},  # 定义五节点分支树训练负例。
    {"id": "G8", "node_count": 5, "edges": [(0, 1), (0, 2), (2, 3), (2, 4)], "label": 0, "split": "train", "说明": "正常双分支"},  # 定义另一棵五节点树训练负例。
    {"id": "G9", "node_count": 6, "edges": [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 0), (0, 3)], "label": 1, "split": "structure_test", "说明": "未见六节点环加跨边"},  # 定义节点数和拓扑均未见的正例。
    {"id": "G10", "node_count": 7, "edges": [(0, 1), (1, 2), (2, 0), (2, 3), (3, 4), (4, 5), (5, 3), (5, 6)], "label": 1, "split": "structure_test", "说明": "未见七节点双环桥接"},  # 定义两个环通过桥连接的新正例。
    {"id": "G11", "node_count": 6, "edges": [(0, 1), (1, 2), (2, 3), (3, 4), (2, 5)], "label": 0, "split": "structure_test", "说明": "未见六节点长链分支树"},  # 定义六节点无环新负例。
    {"id": "G12", "node_count": 7, "edges": [(0, 1), (0, 2), (1, 3), (1, 4), (2, 5), (2, 6)], "label": 0, "split": "structure_test", "说明": "未见七节点平衡树"},  # 定义七节点平衡树新负例。
]  # 结束图规格列表。
def adjacency_from_edges(edges, node_count):  # 定义不依赖高层图包的变长邻接矩阵构造。
    adjacency = torch.zeros((node_count, node_count), dtype=torch.float32)  # 按当前节点数创建方阵。
    for left, right in edges:  # 遍历当前图全部无向边。
        adjacency[left, right] = 1.0  # 写入正向连接。
        adjacency[right, left] = 1.0  # 写入反向连接。
    return adjacency  # 返回无自环邻接矩阵。
graphs = [{**spec, "adjacency": adjacency_from_edges(spec["edges"], spec["node_count"]), "features": torch.ones((spec["node_count"], 1), dtype=torch.float32)} for spec in graph_specs]  # 为每张变长图创建同质节点特征与邻接矩阵。
print("账户子图输入预览：")  # 输出真实案例标题。
pprint([{"图": graph["id"], "说明": graph["说明"], "节点数": graph["node_count"], "边": graph["edges"], "度数": graph["adjacency"].sum(dim=1).tolist(), "label": graph["label"], "split": graph["split"]} for graph in graphs])  # 展示训练图与未见尺寸结构测试图。

账户子图输入预览：
[{'label': 1,
  'split': 'train',
  '图': 'G1',
  '度数': [2.0, 2.0, 3.0, 2.0, 1.0],
  '节点数': 5,
  '说明': '三账户闭环带尾链',
  '边': [(0, 1), (1, 2), (2, 0), (2, 3), (3, 4)]},
 {'label': 1,
  'split': 'train',
  '图': 'G2',
  '度数': [2.0, 2.0, 2.0, 3.0, 1.0],
  '节点数': 5,
  '说明': '四账户闭环',
  '边': [(0, 1), (1, 2), (2, 3), (3, 0), (3, 4)]},
 {'label': 1,
  'split': 'train',
  '图': 'G3',
  '度数': [2.0, 2.0, 2.0, 2.0, 2.0],
  '节点数': 5,
  '说明': '五账户闭环',
  '边': [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0)]},
 {'label': 1,
  'split': 'train',
  '图': 'G4',
  '度数': [4.0, 2.0, 2.0, 1.0, 1.0],
  '节点数': 5,
  '说明': '三角团伙带双叶',
  '边': [(0, 1), (1, 2), (2, 0), (0, 3), (0, 4)]},
 {'label': 0,
  'split': 'train',
  '图': 'G5',
  '度数': [1.0, 2.0, 2.0, 2.0, 1.0],
  '节点数': 5,
  '说明': '正常链路',
  '边': [(0, 1), (1, 2), (2, 3), (3, 4)]},
 {'label': 0,
  'split': 'train',
  '图': 'G6',
  '度数': [4.0, 1.0, 1.0, 1.0, 1.0],
  '节点数': 5,
  '说明': '正常星形',
  '边': [(0, 1), (0, 2), (0, 3), (0, 4)]},
 {'label': 0,
  'split': 'train',
  '

## Baseline / 基线：只平均原始节点特征

所有节点原始特征均为 1，无论图有 5、6 还是 7 个节点，mean readout 都等于 1；平衡结构泛化集上的固定正常类准确率为 0.5。这个 baseline 和 GIN 使用相同的四张新尺寸图、相同 accuracy 指标。

In [2]:
baseline_rows = []  # 创建逐结构泛化图基线结果表。
for graph in graphs:  # 遍历全部账户子图。
    if graph["split"] == "structure_test":  # 只评估未参与训练的六/七节点图。
        graph_mean = float(graph["features"].mean())  # 计算不同尺寸下仍完全相同的原始节点均值。
        prediction = 0  # 平衡数据下使用固定正常类作为基线。
        baseline_rows.append({"图": graph["id"], "节点数": graph["node_count"], "原始图均值": graph_mean, "真实": graph["label"], "预测": prediction})  # 保存同质特征和基线预测。
baseline_accuracy = sum(row["真实"] == row["预测"] for row in baseline_rows) / len(baseline_rows)  # 计算结构泛化集准确率。
print("原始节点均值 Baseline：")  # 输出基线结果标题。
pprint(baseline_rows)  # 展示变长图仍得到同一个无拓扑输入。
print(f"Baseline structure-test accuracy={baseline_accuracy:.3f}")  # 输出后续 GIN 的同指标参照。

原始节点均值 Baseline：
[{'原始图均值': 1.0, '图': 'G9', '真实': 1, '节点数': 6, '预测': 0},
 {'原始图均值': 1.0, '图': 'G10', '真实': 1, '节点数': 7, '预测': 0},
 {'原始图均值': 1.0, '图': 'G11', '真实': 0, '节点数': 6, '预测': 0},
 {'原始图均值': 1.0, '图': 'G12', '真实': 0, '节点数': 7, '预测': 0}]
Baseline structure-test accuracy=0.500


## 手写两层 GIN 与变长图 mean readout

每层先用 `adjacency @ h` 得到邻居求和，再和可学习 epsilon 缩放的自身表示组合，最后经过手写两层 MLP。mean readout 对每张图自己的全部节点池化，因此同一个模型可接收 5、6、7 节点图；这避免仅凭节点数量分类，但并不保证结构泛化一定成功。

In [3]:
class GINLayer(torch.nn.Module):  # 定义单层 GIN 消息传递
    def __init__(self, input_size, hidden_size):  # 初始化 epsilon 与两层 MLP 参数
        super().__init__()  # 初始化 PyTorch 模块基类
        self.epsilon = torch.nn.Parameter(torch.tensor(0.0))  # 创建可学习自身残差系数
        self.first_weight = torch.nn.Parameter(torch.randn(input_size, hidden_size) * 0.3)  # 创建第一层 MLP 权重
        self.first_bias = torch.nn.Parameter(torch.zeros(hidden_size))  # 创建第一层 MLP 偏置
        self.second_weight = torch.nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.3)  # 创建第二层 MLP 权重
        self.second_bias = torch.nn.Parameter(torch.zeros(hidden_size))  # 创建第二层 MLP 偏置
    def forward(self, features, adjacency):  # 定义 GIN sum 聚合与 MLP 前向过程
        neighbor_sum = adjacency @ features  # 对每个节点求邻居表示之和
        aggregated = (1.0 + self.epsilon) * features + neighbor_sum  # 融合自身表示与邻居多重集合
        hidden = torch.relu(aggregated @ self.first_weight + self.first_bias)  # 运行第一层非线性映射
        output = torch.relu(hidden @ self.second_weight + self.second_bias)  # 运行第二层非线性映射
        return output, neighbor_sum, aggregated  # 返回节点表示与关键聚合中间量
class TinyGINClassifier(torch.nn.Module):  # 定义两层 GIN 图分类器
    def __init__(self, hidden_size=12):  # 初始化两层消息传递和分类头
        super().__init__()  # 初始化 PyTorch 模块基类
        self.layer1 = GINLayer(1, hidden_size)  # 创建从一维节点特征到隐藏空间的第一层
        self.layer2 = GINLayer(hidden_size, hidden_size)  # 创建第二层拓扑消息传播
        self.classifier_weight = torch.nn.Parameter(torch.randn(hidden_size, 2) * 0.2)  # 创建图 embedding 到两类 logits 的权重
        self.classifier_bias = torch.nn.Parameter(torch.zeros(2))  # 创建图分类偏置
    def forward(self, graph_batch):  # 定义不同邻接矩阵图列表的批次前向
        graph_embeddings = []  # 收集每张图的池化表示
        details = []  # 收集节点和聚合中间量
        for graph in graph_batch:  # 逐张处理小图而不调用图框架
            hidden1, neighbor_sum1, aggregated1 = self.layer1(graph["features"], graph["adjacency"])  # 执行第一层 GIN
            hidden2, neighbor_sum2, aggregated2 = self.layer2(hidden1, graph["adjacency"])  # 执行第二层 GIN
            graph_embedding = hidden2.mean(dim=0)  # 对当前图全部节点做 mean readout
            graph_embeddings.append(graph_embedding)  # 保存当前图 embedding
            details.append({"hidden1": hidden1, "hidden2": hidden2, "neighbor_sum1": neighbor_sum1, "aggregated1": aggregated1})  # 保存可解释中间张量
        stacked_embeddings = torch.stack(graph_embeddings)  # 把图向量堆叠成批次矩阵
        logits = stacked_embeddings @ self.classifier_weight + self.classifier_bias  # 输出每张图的两类 logits
        return logits, stacked_embeddings, details  # 返回分类结果、图向量和节点过程
model = TinyGINClassifier()  # 实例化手写 GIN 图分类器
with torch.no_grad():  # 关闭结构检查阶段梯度记录
    initial_logits, initial_embeddings, initial_details = model(graphs[:2])  # 对两张图运行未训练前向
print("GIN 张量 shape：", {"logits": tuple(initial_logits.shape), "graph_embeddings": tuple(initial_embeddings.shape), "node_hidden": tuple(initial_details[0]["hidden2"].shape)})  # 展示节点到图级张量形状
print("G1 第一层邻居和与聚合值：", {"neighbor_sum": initial_details[0]["neighbor_sum1"].flatten().tolist(), "aggregated": [round(float(value), 3) for value in initial_details[0]["aggregated1"].flatten()]})  # 展示度数信息如何进入 GIN

GIN 张量 shape： {'logits': (2, 2), 'graph_embeddings': (2, 12), 'node_hidden': (5, 12)}
G1 第一层邻居和与聚合值： {'neighbor_sum': [2.0, 2.0, 3.0, 2.0, 1.0], 'aggregated': [3.0, 3.0, 4.0, 3.0, 2.0]}


## 五节点图训练与真实 backward

loss 只在 8 张五节点训练图上计算，四张六/七节点结构测试图从不参与参数更新。这里使用基础 Adam 更新手写层参数，并记录第一层权重梯度。

In [4]:
train_graphs = [graph for graph in graphs if graph["split"] == "train"]  # 按整图切出八张训练样本
train_targets = torch.tensor([graph["label"] for graph in train_graphs], dtype=torch.long)  # 构造训练图标签张量
optimizer = torch.optim.Adam(model.parameters(), lr=0.015)  # 使用基础 Adam 更新手写 GIN 参数
training_ledger = []  # 创建 loss、准确率与梯度账本
for epoch in range(501):  # 执行五百零一次整图训练
    optimizer.zero_grad()  # 清空上一轮梯度
    logits, graph_embeddings, details = model(train_graphs)  # 对八张训练图运行真实 forward
    log_probabilities = logits - torch.logsumexp(logits, dim=1, keepdim=True)  # 手写两类对数 softmax
    loss = -log_probabilities[torch.arange(len(train_graphs)), train_targets].mean()  # 计算训练图平均交叉熵
    loss.backward()  # 运行真实 backward 计算消息层梯度
    gradient_norm = float(model.layer1.first_weight.grad.norm())  # 读取第一层 MLP 梯度范数
    train_accuracy = float((logits.argmax(dim=1) == train_targets).to(torch.float32).mean())  # 计算当前训练图准确率
    if epoch % 100 == 0:  # 每一百轮记录一次训练状态
        training_ledger.append({"epoch": epoch, "loss": round(float(loss), 6), "train_accuracy": round(train_accuracy, 3), "first_layer_grad": round(gradient_norm, 6)})  # 保存真实损失和梯度
    optimizer.step()  # 根据当前梯度更新全部手写参数
print("GIN 训练 loss、准确率与梯度：")  # 输出训练过程标题
pprint(training_ledger)  # 展示图分类学习过程

GIN 训练 loss、准确率与梯度：
[{'epoch': 0,
  'first_layer_grad': 0.379575,
  'loss': 0.733719,
  'train_accuracy': 0.5},
 {'epoch': 100,
  'first_layer_grad': 3.183581,
  'loss': 0.085476,
  'train_accuracy': 1.0},
 {'epoch': 200,
  'first_layer_grad': 0.050747,
  'loss': 0.012945,
  'train_accuracy': 1.0},
 {'epoch': 300,
  'first_layer_grad': 0.016957,
  'loss': 0.001763,
  'train_accuracy': 1.0},
 {'epoch': 400,
  'first_layer_grad': 0.911143,
  'loss': 0.010188,
  'train_accuracy': 1.0},
 {'epoch': 500,
  'first_layer_grad': 0.005075,
  'loss': 0.001198,
  'train_accuracy': 1.0}]


## 评估一：未见尺寸与未见结构的泛化结果与结果解读

这四张图不是训练图的节点重标号：训练图全部 5 节点，评估图明确为 6/7 节点，并包含六环加跨边、双环桥接、长链分支树和平衡树。下面逐图诚实报告预测；无论结果高低，都只能说明这组受控结构测试，不能外推为任意图泛化能力。

In [5]:
train_node_counts = sorted({graph["node_count"] for graph in graphs if graph["split"] == "train"})  # 汇总训练阶段见过的图尺寸。
test_graphs = [graph for graph in graphs if graph["split"] == "structure_test"]  # 取出四张未见尺寸和拓扑的结构测试图。
test_node_counts = sorted({graph["node_count"] for graph in test_graphs})  # 汇总结构测试图尺寸。
with torch.no_grad():  # 关闭结构泛化评估阶段梯度记录。
    test_logits, test_embeddings, test_details = model(test_graphs)  # 运行同一个 GIN 的变长图前向。
    shifted = test_logits - test_logits.max(dim=1, keepdim=True).values  # 稳定两类 softmax 输入。
    test_probabilities = shifted.exp() / shifted.exp().sum(dim=1, keepdim=True)  # 手写测试图类别概率。
    test_predictions = test_probabilities.argmax(dim=1)  # 用预先固定的 argmax 产生类别决策。
test_targets = torch.tensor([graph["label"] for graph in test_graphs], dtype=torch.long)  # 构造结构测试图真实标签。
gin_test_accuracy = float((test_predictions == test_targets).to(torch.float32).mean())  # 计算未见结构 accuracy。
result_rows = []  # 创建逐结构测试图结果表。
for index, graph in enumerate(test_graphs):  # 遍历四张新尺寸图。
    result_rows.append({"图": graph["id"], "说明": graph["说明"], "节点数": graph["node_count"], "度序列": sorted(graph["adjacency"].sum(dim=1).tolist()), "真实": graph["label"], "Baseline": 0, "GIN": int(test_predictions[index]), "团伙概率": round(float(test_probabilities[index, 1]), 4), "embedding前4维": [round(float(value), 3) for value in test_embeddings[index, :4]]})  # 保存结构证据、预测概率和图向量片段。
print(f"训练尺寸={train_node_counts}，结构测试尺寸={test_node_counts}，二者不重叠。")  # 明确排除同尺寸节点重标号冒充泛化。
print("未见尺寸/结构逐图结果：")  # 输出结构泛化结果标题。
pprint(result_rows)  # 展示四种新结构的真实预测。
print(f"结构泛化 accuracy：Baseline={baseline_accuracy:.3f}，GIN={gin_test_accuracy:.3f}")  # 诚实输出同一结构测试集上的比较。

训练尺寸=[5]，结构测试尺寸=[6, 7]，二者不重叠。
未见尺寸/结构逐图结果：
[{'Baseline': 0,
  'GIN': 1,
  'embedding前4维': [0.0, 0.0, 3.862, 4.473],
  '团伙概率': 1.0,
  '图': 'G9',
  '度序列': [2.0, 2.0, 2.0, 2.0, 3.0, 3.0],
  '真实': 1,
  '节点数': 6,
  '说明': '未见六节点环加跨边'},
 {'Baseline': 0,
  'GIN': 1,
  'embedding前4维': [0.0, 0.248, 3.83, 4.522],
  '团伙概率': 1.0,
  '图': 'G10',
  '度序列': [1.0, 2.0, 2.0, 2.0, 3.0, 3.0, 3.0],
  '真实': 1,
  '节点数': 7,
  '说明': '未见七节点双环桥接'},
 {'Baseline': 0,
  'GIN': 0,
  'embedding前4维': [0.0, 0.946, 1.463, 1.546],
  '团伙概率': 0.0031,
  '图': 'G11',
  '度序列': [1.0, 1.0, 1.0, 2.0, 2.0, 3.0],
  '真实': 0,
  '节点数': 6,
  '说明': '未见六节点长链分支树'},
 {'Baseline': 0,
  'GIN': 0,
  'embedding前4维': [0.0, 0.993, 1.615, 1.721],
  '团伙概率': 0.005,
  '图': 'G12',
  '度序列': [1.0, 1.0, 1.0, 1.0, 2.0, 3.0, 3.0],
  '真实': 0,
  '节点数': 7,
  '说明': '未见七节点平衡树'}]
结构泛化 accuracy：Baseline=0.500，GIN=1.000


## 评估二：同一张图的节点置换不变性

这一项只回答“节点编号改变是否影响图级输出”，不回答结构泛化。对 G9 使用固定 permutation 同时重排邻接矩阵的行列和节点特征，再比较原图与重编号图的 embedding、logits 和预测。

In [6]:
reference_graph = test_graphs[0]  # 选择六节点新结构 G9 作为置换对象。
permutation = torch.tensor([2, 5, 0, 4, 1, 3], dtype=torch.long)  # 定义固定的新行到旧节点编号映射。
permuted_graph = {**reference_graph, "id": "G9-permuted", "features": reference_graph["features"][permutation], "adjacency": reference_graph["adjacency"][permutation][:, permutation]}  # 同步重排特征和邻接行列以保持同一抽象图。
with torch.no_grad():  # 关闭置换检查的梯度记录。
    permutation_logits, permutation_embeddings, permutation_details = model([reference_graph, permuted_graph])  # 在同一个 batch 中编码原图与重编号图。
    permutation_probabilities = torch.softmax(permutation_logits, dim=1)  # 计算两份图的类别概率。
permutation_embedding_difference = float((permutation_embeddings[0] - permutation_embeddings[1]).abs().max())  # 计算图 embedding 最大绝对差。
permutation_logit_difference = float((permutation_logits[0] - permutation_logits[1]).abs().max())  # 计算分类 logits 最大绝对差。
permutation_predictions = permutation_probabilities.argmax(dim=1)  # 读取两份表示的 argmax 预测。
print("置换映射 new_row->old_node=", permutation.tolist())  # 展示可复现节点重编号。
print("原图/置换图团伙概率=", [round(float(value), 6) for value in permutation_probabilities[:, 1]])  # 展示同一图两次前向的概率。
print(f"置换不变性：embedding最大差={permutation_embedding_difference:.8f}，logit最大差={permutation_logit_difference:.8f}，预测={permutation_predictions.tolist()}")  # 量化模型对节点编号的不敏感性。
print("解释：这里的近零差异只证明置换不变性；上一个四图 accuracy 才是本例的结构泛化证据。")  # 明确区分两个评估问题。

置换映射 new_row->old_node= [2, 5, 0, 4, 1, 3]
原图/置换图团伙概率= [1.0, 1.0]
置换不变性：embedding最大差=0.00000048，logit最大差=0.00000095，预测=[1, 1]
解释：这里的近零差异只证明置换不变性；上一个四图 accuracy 才是本例的结构泛化证据。


## 失败案例：mean 聚合丢失邻居重复次数

中心节点接收一个特征为 1 的邻居和两个都为 1 的邻居时，mean 都等于 1，无法区分多重集合；sum 分别为 1 和 2。GIN 保留这个计数差异，后续 MLP 才可能学习结构模式。

In [7]:
one_neighbor = torch.tensor([[1.0]], dtype=torch.float32)  # 构造只有一个相同特征邻居的集合
two_neighbors = torch.tensor([[1.0], [1.0]], dtype=torch.float32)  # 构造包含两个重复邻居的集合
unsafe_mean_one = one_neighbor.mean(dim=0)  # 对一个邻居执行 mean 聚合
unsafe_mean_two = two_neighbors.mean(dim=0)  # 对两个相同邻居执行 mean 聚合
safe_sum_one = one_neighbor.sum(dim=0)  # 对一个邻居执行 sum 聚合
safe_sum_two = two_neighbors.sum(dim=0)  # 对两个相同邻居执行 sum 聚合
print("失败案例：mean 聚合", {"一个邻居": unsafe_mean_one.tolist(), "两个邻居": unsafe_mean_two.tolist()})  # 展示 mean 无法区分重复次数
print("GIN 修正：sum 聚合", {"一个邻居": safe_sum_one.tolist(), "两个邻居": safe_sum_two.tolist()})  # 展示 sum 保留多重集合计数

失败案例：mean 聚合 {'一个邻居': [1.0], '两个邻居': [1.0]}
GIN 修正：sum 聚合 {'一个邻居': [1.0], '两个邻居': [2.0]}


## 生产差距

真实图分类需要节点/边多类型特征、mini-batch 图拼接、稀疏算子、邻居采样和严格时间切分。置换不变性是架构性质，不等于结构、尺寸、时间或业务域泛化；生产评测应按节点数、motif、时间和实体群组分别构造真正未见的测试桶。GIN 仍无法区分所有非同构图，层数增加还会过平滑；线上风控还需防同实体跨切分泄漏、处理类别不平衡、概率校准、解释审计和结构漂移。

In [8]:
assert len(graphs) == 12 and len(test_graphs) == 4  # 验证案例包含八张训练图和四张结构测试图。
assert training_ledger[-1]["loss"] < training_ledger[0]["loss"]  # 验证真实 backward 降低图分类损失。
assert train_node_counts == [5] and test_node_counts == [6, 7] and set(train_node_counts).isdisjoint(test_node_counts)  # 验证结构测试不可能只是五节点训练图重标号。
assert test_predictions.shape[0] == len(test_graphs) and 0.0 <= gin_test_accuracy <= 1.0  # 验证诚实保存四张新结构预测与合法指标。
assert permutation_embedding_difference < 1.0e-5 and permutation_logit_difference < 1.0e-5 and permutation_predictions[0] == permutation_predictions[1]  # 验证同一图节点置换不改变图级输出。
assert torch.equal(unsafe_mean_one, unsafe_mean_two) and not torch.equal(safe_sum_one, safe_sum_two)  # 验证 mean 失败而 sum 保留邻居计数。
print("最小回归测试通过：真实训练、未见尺寸结构评估、置换不变性与 mean 反例均已分别验证。")  # 输出集中断言的验收结论。

最小回归测试通过：真实训练、未见尺寸结构评估、置换不变性与 mean 反例均已分别验证。
